In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Figure 14.15: Activations before and after applying LRN

**Local Response Normalization (LRN)** normalizes each neuron's activation using the responses of neighboring feature channels at the same spatial location.

The formula is:
$$b_i = \frac{a_i}{\left( k + \alpha \sum_{j=\max(0,i-\lfloor n/2\rfloor)}^{\min(N-1,i+\lfloor n/2\rfloor)} a_j^2 \right)^\beta}$$

Large activations are suppressed more than small ones, encouraging **competition across channels** and preventing any single feature map from dominating.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def lrn(a, n, k, alpha, beta):
    """Local Response Normalization across channels."""
    N = len(a)
    half = n // 2
    b = np.zeros(N)
    for i in range(N):
        j_lo = max(0, i - half)
        j_hi = min(N - 1, i + half)
        denom = (k + alpha * np.sum(a[j_lo:j_hi + 1] ** 2)) ** beta
        b[i] = a[i] / denom
    return b


# Fixed toy activations for a single spatial location
BASE_ACTS = np.array([3.5, 1.2, 4.8, 0.7, 2.9, 1.5, 3.1, 0.4])


def draw_lrn(N=4, n=2, k=0.0, alpha=1.0, beta=1.0):
    a = BASE_ACTS[:N]
    b = lrn(a, n, k, alpha, beta)
    channels = [f'Ch {i}' for i in range(N)]
    colors = ['#2563eb', '#dc2626', '#059669', '#d97706', '#7c3aed', '#0891b2', '#be185d', '#65a30d']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar(channels, a, color=colors[:N], edgecolor='white', linewidth=1.5)
    axes[0].set_ylim(0, 6)
    axes[0].set_title('Activations before LRN', fontsize=11)
    axes[0].set_ylabel('Activation value')
    axes[0].grid(True, axis='y', alpha=0.3)
    for i, v in enumerate(a):
        axes[0].text(i, v + 0.08, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')

    axes[1].bar(channels, b, color=colors[:N], alpha=0.8, edgecolor='white', linewidth=1.5)
    axes[1].set_ylim(0, max(b) * 1.3 + 0.1)
    axes[1].set_title(f'Activations after LRN  (k={k}, α={alpha}, β={beta}, n={n})', fontsize=11)
    axes[1].set_ylabel('Normalized activation')
    axes[1].grid(True, axis='y', alpha=0.3)
    for i, v in enumerate(b):
        axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

    half = n // 2
    print(f'\nLRN window half = {half}  |  k={k}, α={alpha}, β={beta}\n')
    for i in range(N):
        j_lo = max(0, i - half)
        j_hi = min(N - 1, i + half)
        denom = (k + alpha * np.sum(a[j_lo:j_hi + 1] ** 2)) ** beta
        print(f'  Ch {i}: a={a[i]:.2f}  window=[{j_lo}..{j_hi}]  Σa²={np.sum(a[j_lo:j_hi+1]**2):.2f}  denom={denom:.3f}  b={b[i]:.4f}')


widgets.interact(
    draw_lrn,
    N=widgets.IntSlider(min=3, max=8, step=1, value=4, description='Channels N', continuous_update=False),
    n=widgets.IntSlider(min=1, max=5, step=1, value=2, description='Window n', continuous_update=False),
    k=widgets.FloatSlider(min=0, max=4, step=0.5, value=0.0, description='k (bias)', continuous_update=False),
    alpha=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='α', continuous_update=False),
    beta=widgets.FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, description='β', continuous_update=False),
)